# Inspect cleaned replay files

Browse outputs under **`data/output/clean`** using pandas and PyArrow, without Spark or Java.
This notebook reads files only. It does not rerun the cleaner, change tables, or resolve outstanding
kill/assist credit. The [cleaning walkthrough](cleaning_walkthrough.ipynb) explains the transformations.

Install dependencies using `python -m pip install -r notebooks/requirements.txt` from the repository
root, select that Python environment, and run cells in order. Set `FOCUS_MATCH_ID` below to choose
a match. Completion status and candidate status are shown explicitly so a preview cannot be
mistaken for warehouse-ready output.

In [1]:
from pathlib import Path
import json
import re
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "warehouse" / "init" / "02_clean_tables.sql").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the dota-pipeline repository.")
CLEAN_DIR = ROOT / "data" / "output" / "clean"
FOCUS_MATCH_ID = 9079737909  # Use None to select the first discovered match.
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 100)

# Read expected columns, types, and keys from the actual warehouse DDL, without importing Spark.
SQL_TYPES = {"BIGINT": pa.int64(), "INTEGER": pa.int32(), "DOUBLE PRECISION": pa.float64(),
             "TEXT": pa.string(), "BOOLEAN": pa.bool_()}
specs = {}
sql = (ROOT / "warehouse" / "init" / "02_clean_tables.sql").read_text()
for table, body in re.findall(r"CREATE TABLE IF NOT EXISTS clean\.(\w+)\s*\((.*?)\);", sql, flags=re.S):
    columns = []; inline_keys = []
    for line in body.splitlines():
        match = re.match(r"\s*(\w+)\s+(BIGINT|INTEGER|DOUBLE PRECISION|TEXT|BOOLEAN)\b(.*)", line)
        if match:
            name, dtype, constraints = match.groups()
            columns.append({"column": name, "sql_type": dtype,
                            "required": "NOT NULL" in constraints or "PRIMARY KEY" in constraints})
            if "PRIMARY KEY" in constraints:
                inline_keys.append(name)
    explicit_key = re.search(r"PRIMARY KEY\s*\(([^)]+)\)", body)
    key = [c.strip() for c in explicit_key.group(1).split(",")] if explicit_key else inline_keys
    for column in columns:
        column["required"] = column["required"] or column["column"] in key
    specs[table] = {"columns": columns, "key": key}
if len(specs) != 9:
    raise RuntimeError("Expected nine warehouse clean-table definitions; inspect the DDL parser.")
OPTIONAL_EMPTY = {"purchases", "death_assists"}

## 1. Find outputs and check completion status

A cleaner report can describe candidate counts even when the corresponding final table was not
published. The completion marker, report errors, and preview flag must all agree before treating
a run as ready. This section discovers written tables as well as review-only matches.

In [2]:
match_ids = set()
for folder in [*(CLEAN_DIR / t for t in specs), CLEAN_DIR / "_review", CLEAN_DIR / "_complete"]:
    for p in folder.glob("match_id=*"):
        suffix = p.name.split("=", 1)[1]
        if suffix.isdigit():
            match_ids.add(int(suffix))
match_ids = sorted(match_ids)
if not match_ids:
    raise RuntimeError(f"No clean outputs found under {CLEAN_DIR}. Run the cleaner first.")
reports = {}; run_rows = []
for mid in match_ids:
    path = CLEAN_DIR / "_review" / f"match_id={mid}" / "report.json"
    report = json.loads(path.read_text()) if path.exists() else None
    if report is not None and report.get("match_id") != mid:
        raise ValueError(f"Report match ID disagrees with its folder: {path}")
    reports[mid] = report
    marker = CLEAN_DIR / "_complete" / f"match_id={mid}"
    marker_present = marker.exists() and any(p.is_file() for p in marker.rglob("*"))
    ready = bool(marker_present and report and report.get("complete")
                 and not report.get("preview") and not report.get("errors"))
    run_rows.append({"match_id": mid, "report_found": report is not None,
        "preview": report.get("preview") if report else None,
        "report_complete": report.get("complete") if report else None,
        "marker_present": marker_present, "ready": ready,
        "errors": "; ".join(report.get("errors", [])) if report else "Report missing"})
run_status = pd.DataFrame(run_rows)
display(run_status)
MATCH_ID = FOCUS_MATCH_ID if FOCUS_MATCH_ID is not None else match_ids[0]
if MATCH_ID not in match_ids:
    raise ValueError(f"Match {MATCH_ID} has no output. Choose one from {match_ids}.")
report = reports[MATCH_ID]
print("Selected match:", MATCH_ID)
print("Report:", CLEAN_DIR / "_review" / f"match_id={MATCH_ID}" / "report.json")

,match_id,report_found,preview,report_complete,marker_present,ready,errors
0,9079737909,True,True,False,False,False,scoreboard_disagreements: 1 rows need review


Selected match: 9079737909
Report: /Users/maximus/Downloads/SCHOOL/IS3107/dota-pipeline/data/output/clean/_review/match_id=9079737909/report.json


## 2. Load tables and distinguish written output from review candidates

Parquet files physically omit `match_id` because it is in the partition path. The reader restores
it as a 64-bit integer. Reading physical files directly avoids accidentally mixing ancestor and
candidate partition names. Expected empty optional tables are represented as empty typed frames;
missing required tables are flagged, not silently assumed valid.

In [3]:
def read_parts(folder, match_id=None):
    files = sorted(folder.glob("*.parquet"))
    if not files:
        return None
    parts = []
    for path in files:
        frame = pq.ParquetFile(path).read().to_pandas().convert_dtypes()
        if match_id is not None:
            if "match_id" in frame and not frame["match_id"].eq(match_id).all():
                raise ValueError(f"Unexpected embedded match ID: {path}")
            frame["match_id"] = pd.Series(match_id, index=frame.index, dtype="Int64")
        parts.append(frame)
    return pd.concat(parts, ignore_index=True)

def empty_table(table):
    schema = pa.schema([(c["column"], SQL_TYPES[c["sql_type"]]) for c in specs[table]["columns"]])
    return pa.Table.from_batches([], schema=schema).to_pandas().convert_dtypes()

tables = {}; sources = {}; folders = {}; table_rows = []
for table in specs:
    regular = CLEAN_DIR / table / f"match_id={MATCH_ID}"
    candidate = CLEAN_DIR / "_review" / f"match_id={MATCH_ID}" / "candidates" / table / f"match_id={MATCH_ID}"
    expected = report.get("table_rows", {}).get(table) if report else None
    df = read_parts(regular, MATCH_ID)
    source = "written table"; folder = regular
    if df is None:
        df = read_parts(candidate, MATCH_ID)
        source = "CANDIDATE - review only"; folder = candidate
    if df is None and table in OPTIONAL_EMPTY and expected == 0:
        df = empty_table(table)
        source = "empty candidate" if table == "death_assists" and not bool(report.get("complete")) else "empty optional table"
    if df is None:
        source = "MISSING"
    else:
        tables[table] = df; sources[table] = source; folders[table] = folder
    table_rows.append({"table": table, "source": source,
        "loaded_rows": len(df) if df is not None else None, "reported_rows": expected,
        "count_matches_report": len(df) == expected if df is not None and expected is not None else None})
table_status = pd.DataFrame(table_rows)
display(table_status)
print("Written tables in a preview/incomplete run still have no completion guarantee.")

,table,source,loaded_rows,reported_rows,count_matches_report
0,matches,written table,1,1,True
1,players,written table,10,10,True
2,player_stats,written table,15080,15080,True
3,positions,written table,14570,14570,True
4,damage_events,written table,8783,8783,True
5,deaths,CANDIDATE - review only,37,37,True
6,death_assists,CANDIDATE - review only,28,28,True
7,purchases,written table,200,200,True
8,final_inventory,written table,75,75,True


Written tables in a preview/incomplete run still have no completion guarantee.


## 3. First five rows of every available table

Candidate deaths and assists are explicitly labelled in each preview. Change the selected match
in setup and rerun to inspect another run.

In [4]:
for table, df in tables.items():
    print(f"\n{table}: {sources[table]} | {len(df):,} rows")
    order = [c for c in ["tick", "player_slot", "victim_slot", "inventory_slot"] if c in df]
    display((df.sort_values(order, kind="stable") if order else df).head(5))


matches: written table | 1 rows


,game_mode,winner_team,end_time,game_start_tick,duration_s,match_id
0,22,3,1794045663,4260,1222.066711,9079737909



players: written table | 10 rows


,player_slot,team,team_slot,hero_name,hero_id,match_id
0,0,2,0,npc_dota_hero_juggernaut,8,9079737909
1,1,2,1,npc_dota_hero_legion_commander,104,9079737909
2,2,2,2,npc_dota_hero_tidehunter,29,9079737909
3,3,2,3,npc_dota_hero_axe,2,9079737909
4,4,2,4,npc_dota_hero_sniper,35,9079737909



player_stats: written table | 15,080 rows


,player_slot,tick,game_time_s,level,kills,deaths,assists,net_worth,unspent_gold,total_earned_gold,total_earned_xp,last_hits,denies,hero_damage,tower_damage,healing,match_id
0,0,30,-141,0,0,0,0,0,600,0,0,0,0,0,0,0,9079737909
1,1,30,-141,0,0,0,0,0,600,0,0,0,0,0,0,0,9079737909
2,2,30,-141,0,0,0,0,0,600,0,0,0,0,0,0,0,9079737909
3,3,30,-141,0,0,0,0,0,600,0,0,0,0,0,0,0,9079737909
4,4,30,-141,0,0,0,0,0,600,0,0,0,0,0,0,0,9079737909


... (8 outputs left out)


purchases: written table | 200 rows


,tick,game_time_s,player_slot,item_name,quantity,match_id
21,4256,-0.133333,1,item_quelling_blade,1,9079737909
45,4256,-0.133333,3,item_quelling_blade,1,9079737909
14,4256,-0.133333,4,item_tango,1,9079737909
76,4258,-0.066667,0,item_quelling_blade,1,9079737909
77,4258,-0.066667,5,item_tango,1,9079737909



final_inventory: written table | 75 rows


,player_slot,inventory_slot,item_name,match_id
53,0,0,item_bfury,9079737909
0,0,1,item_magic_wand,9079737909
12,0,2,item_foragers_mana,9079737909
1,0,3,item_power_treads,9079737909
34,0,4,item_wraith_band,9079737909


## 4. Column types, missing required fields, and repeated keys

Checks use the warehouse DDL. Optional null fields, such as a non-player killer's slot, are allowed.
For nonempty files, types are checked against the physical Arrow schema before pandas conversion.
Structural checks do not establish whether candidate kill/assist credit is correct.

In [5]:
checks = []; type_rows = []
for table, df in tables.items():
    spec = specs[table]
    expected_cols = [c["column"] for c in spec["columns"]]
    missing = sorted(set(expected_cols) - set(df.columns))
    key_present = all(c in df for c in spec["key"])
    required = [c["column"] for c in spec["columns"] if c["required"] and c["column"] in df]
    repeated = df.groupby(spec["key"], dropna=False).size().gt(1).sum() if key_present else None
    checks.append({"table": table, "source": sources[table], "missing_columns": ", ".join(missing),
        "rows_with_required_nulls": int(df[required].isna().any(axis=1).sum()),
        "repeated_key_groups": repeated,
        "empty_required_table": df.empty and table not in OPTIONAL_EMPTY})
    for path in sorted(folders[table].glob("*.parquet")):
        schema = pq.ParquetFile(path).schema_arrow
        for c in spec["columns"]:
            name = c["column"]
            if name == "match_id":
                actual = "partition metadata"; ok = name not in schema.names
            else:
                field = schema.field(name) if name in schema.names else None
                actual = str(field.type) if field is not None else "MISSING"
                ok = field is not None and field.type.equals(SQL_TYPES[c["sql_type"]])
            type_rows.append({"table": table, "column": name, "sql_type": c["sql_type"],
                              "physical_type": actual, "matches": ok})
quality = pd.DataFrame(checks)
types = pd.DataFrame(type_rows).drop_duplicates()
display(quality)
print("Physical type mismatches:")
display(types.loc[~types["matches"]] if not types.empty else types)
TABLE_TO_DESCRIBE = "player_stats"
display(types.loc[types["table"].eq(TABLE_TO_DESCRIBE)] if not types.empty else types)

,table,source,missing_columns,rows_with_required_nulls,repeated_key_groups,empty_required_table
0,matches,written table,,0,0,False
1,players,written table,,0,0,False
2,player_stats,written table,,0,0,False
3,positions,written table,,0,0,False
4,damage_events,written table,,0,0,False
5,deaths,CANDIDATE - review only,,0,0,False
6,death_assists,CANDIDATE - review only,,0,0,False
7,purchases,written table,,0,0,False
8,final_inventory,written table,,0,0,False


Physical type mismatches:


,table,column,sql_type,physical_type,matches


,table,column,sql_type,physical_type,matches
12,player_stats,match_id,BIGINT,partition metadata,True
13,player_stats,player_slot,INTEGER,int32,True
14,player_stats,tick,INTEGER,int32,True
15,player_stats,game_time_s,DOUBLE PRECISION,double,True
16,player_stats,level,INTEGER,int32,True
17,player_stats,kills,INTEGER,int32,True
18,player_stats,deaths,INTEGER,int32,True
19,player_stats,assists,INTEGER,int32,True
20,player_stats,net_worth,INTEGER,int32,True
21,player_stats,unspent_gold,INTEGER,int32,True


## 5. Report findings and the actual review rows

`blocking=True` describes what happens if a finding has affected rows; a zero-row blocking check
is a pass. Diagnostic files are only written for nonempty findings. The report's disagreement
count is a number of players, not a number of proven erroneous individual events.

In [6]:
findings = pd.DataFrame([{"finding": name, **details}
    for name, details in (report.get("findings", {}) if report else {}).items()])
if not findings.empty:
    findings["blocks_this_run"] = findings["blocking"] & findings["rows"].gt(0)
display(findings)
review_dir = CLEAN_DIR / "_review" / f"match_id={MATCH_ID}"
REVIEW_FINDING = "scoreboard_disagreements"
review_rows = read_parts(review_dir / REVIEW_FINDING)
if review_rows is None:
    print(f"No saved diagnostic rows for {REVIEW_FINDING}.")
else:
    display(review_rows)

,finding,rows,blocking,blocks_this_run
0,invalid_game_clock,0,True,False
1,excluded_resource_rows,0,False,False
2,duplicate_player_snapshots,0,True,False
3,unmapped_resource_rows,0,True,False
4,duplicate_team_snapshots,0,True,False
5,missing_team_snapshots,0,True,False
6,unknown_replication_marker,0,True,False
7,invalid_hero_player_ids,0,True,False
8,unmapped_hero_teams,0,True,False
9,damage_unknown_illusion_flags,0,True,False


,match_id,player_slot,kills,deaths,assists,event_kills,event_deaths,event_assists
0,9079737909,4,2,8,0,0,8,2


## 6. Compare scoreboard and candidate event counts

Show both counts and differences rather than labelling every related event as incorrect.
Positive differences mean more candidate events than final snapshot credit. Events listed for an
affected player are investigation leads, not proof of the cause. They may be candidates from an
incomplete run, not final warehouse facts.

In [7]:
scoreboard = read_parts(review_dir / "scoreboard_disagreements")
if scoreboard is None:
    print("No saved scoreboard disagreements.")
else:
    comparison = scoreboard.copy()
    for statistic in ["kills", "deaths", "assists"]:
        event_col = "event_" + statistic
        if statistic in comparison and event_col in comparison:
            comparison[statistic + "_difference"] = comparison[event_col] - comparison[statistic]
    if "players" in tables:
        comparison = comparison.merge(tables["players"][["match_id", "player_slot", "hero_name"]],
            on=["match_id", "player_slot"], how="left", validate="many_to_one")
    display(comparison)
    if "death_assists" in tables:
        related_assists = tables["death_assists"].merge(scoreboard[["match_id", "player_slot"]].rename(
            columns={"player_slot": "assist_slot"}), on=["match_id", "assist_slot"], how="inner", validate="many_to_one")
        if "deaths" in tables:
            related_assists = related_assists.merge(tables["deaths"],
                on=["match_id", "tick", "victim_slot"], how="left", validate="many_to_one")
        print("Related assist candidates:")
        display(related_assists.sort_values(["assist_slot", "tick"]))

,match_id,player_slot,kills,deaths,assists,event_kills,event_deaths,event_assists,kills_difference,deaths_difference,assists_difference,hero_name
0,9079737909,4,2,8,0,0,8,2,-2,0,2,npc_dota_hero_sniper


Related assist candidates:


,tick,victim_slot,assist_slot,match_id,game_time_s,killer_slot,killer_name
0,17682,8,4,9079737909,447.4,<NA>,npc_dota_goodguys_tower1_mid
1,19898,8,4,9079737909,521.266667,<NA>,npc_dota_goodguys_tower1_mid


## 7. Final player snapshots and inventory

This is a convenience view of each player's last written stats snapshot, not a recalculation or
correction of kills/assists. Inventory stays in its own slot-level table so empty inventory slots
are not mistaken for missing player records.

In [8]:
if "players" in tables and "player_stats" in tables:
    stats = tables["player_stats"]
    last_ticks = stats.groupby(["match_id", "player_slot"], as_index=False)["tick"].max()
    final_stats = stats.merge(last_ticks, on=["match_id", "player_slot", "tick"], how="inner")
    roster_view = tables["players"].merge(final_stats, on=["match_id", "player_slot"], how="left", validate="one_to_one")
    display(roster_view.sort_values("player_slot"))
if "final_inventory" in tables:
    display(tables["final_inventory"].sort_values(["player_slot", "inventory_slot"]))

,player_slot,team,team_slot,hero_name,hero_id,match_id,tick,game_time_s,level,kills,deaths,assists,net_worth,unspent_gold,total_earned_gold,total_earned_xp,last_hits,denies,hero_damage,tower_damage,healing
0,0,2,0,npc_dota_hero_juggernaut,8,9079737909,45240,1366,11,1,3,5,7800,235,7504,8166,135,0,1808,0,0
1,1,2,1,npc_dota_hero_legion_commander,104,9079737909,45240,1366,9,3,3,4,6020,205,5887,5534,72,0,5173,0,0
2,2,2,2,npc_dota_hero_tidehunter,29,9079737909,45240,1366,8,4,2,3,5162,347,4847,4350,43,0,4434,0,0
3,3,2,3,npc_dota_hero_axe,2,9079737909,45240,1366,10,2,3,4,5985,945,5661,6289,79,0,2000,0,0
4,4,2,4,npc_dota_hero_sniper,35,9079737909,45240,1366,9,2,8,0,5246,81,5084,5731,49,0,2826,0,0
5,5,3,0,npc_dota_hero_drow_ranger,6,9079737909,45240,1366,8,1,4,5,5905,740,5672,4227,45,0,2736,2879,0
6,6,3,1,npc_dota_hero_sven,18,9079737909,45240,1366,8,2,5,2,5264,449,5089,4315,33,0,5096,2139,0
7,7,3,2,npc_dota_hero_centaur,96,9079737909,45240,1366,9,2,3,2,5980,940,5684,5499,43,0,4638,2915,0
8,8,3,3,npc_dota_hero_huskar,59,9079737909,45240,1366,11,6,2,0,6876,386,6526,7543,39,0,4511,5777,0
9,9,3,4,npc_dota_hero_lina,25,9079737909,45240,1366,8,5,4,1,5458,318,5382,4886,18,0,4822,2318,0


,player_slot,inventory_slot,item_name,match_id
53,0,0,item_bfury,9079737909
0,0,1,item_magic_wand,9079737909
12,0,2,item_foragers_mana,9079737909
1,0,3,item_power_treads,9079737909
34,0,4,item_wraith_band,9079737909
...,...,...,...,...
11,9,1,item_travel_boots,9079737909
73,9,2,item_bottle,9079737909
74,9,3,item_magic_wand,9079737909
32,9,4,item_ogre_axe,9079737909


## 8. Browse a table by player and time

Change these controls and rerun the cell. Player columns vary: `player_slot` in stats/positions/
purchases, `attacker_slot` or `target_slot` in damage, `victim_slot`/`killer_slot` in deaths, and
`assist_slot` in assists. Time filters require the corresponding column; assists have raw ticks
but no `game_time_s` column in the warehouse contract.

In [9]:
BROWSE_TABLE = "positions"
PLAYER_COLUMN = "player_slot"
PLAYER_SLOT = 4  # None disables player filtering; None PLAYER_COLUMN as well is allowed.
TICK_MIN = None
TICK_MAX = None
GAME_TIME_MIN = None  # Negative values include pregame; 0 starts at the chosen horn snapshot.
GAME_TIME_MAX = None
SHOW_COLUMNS = None  # None for all columns, or a list of names.
LIMIT = 20

if BROWSE_TABLE not in tables:
    print(f"No loaded table: {BROWSE_TABLE}")
else:
    subset = tables[BROWSE_TABLE]
    if PLAYER_SLOT is not None:
        if PLAYER_COLUMN not in subset:
            raise ValueError(f"Choose a valid player column from {list(subset.columns)}")
        subset = subset.loc[subset[PLAYER_COLUMN].eq(PLAYER_SLOT).fillna(False)]
    for field, lower, upper in [("tick", TICK_MIN, TICK_MAX), ("game_time_s", GAME_TIME_MIN, GAME_TIME_MAX)]:
        if lower is not None or upper is not None:
            if field not in subset:
                raise ValueError(f"{BROWSE_TABLE} has no {field} column")
            if lower is not None:
                subset = subset.loc[subset[field].ge(lower).fillna(False)]
            if upper is not None:
                subset = subset.loc[subset[field].le(upper).fillna(False)]
    if "tick" in subset:
        subset = subset.sort_values("tick", kind="stable")
    print(f"{sources[BROWSE_TABLE]} | {len(subset):,} matching rows; showing up to {LIMIT}")
    display(subset[SHOW_COLUMNS].head(LIMIT) if SHOW_COLUMNS is not None else subset.head(LIMIT))

written table | 1,457 matching rows; showing up to 20


,player_slot,tick,game_time_s,x,y,match_id
5828,4,1560,-90,-7100.0,-6150.0,9079737909
5829,4,1590,-89,-7100.0,-6150.0,9079737909
5830,4,1620,-88,-7100.0,-6150.0,9079737909
5831,4,1650,-87,-7100.0,-6150.0,9079737909
5832,4,1680,-86,-7100.0,-6150.0,9079737909
5833,4,1710,-85,-7100.0,-6150.0,9079737909
5834,4,1740,-84,-7100.0,-6150.0,9079737909
5835,4,1770,-83,-7100.0,-6150.0,9079737909
5836,4,1800,-82,-7100.0,-6150.0,9079737909
5837,4,1830,-81,-7100.0,-6150.0,9079737909


## 9. Full saved report

This is the JSON produced by the cleaner, including errors, candidate counts, and completion flags.
To change cleaning behavior, update the cleaner and rerun it; this inspection notebook leaves the
saved files and completion marker unchanged.

In [10]:
print(json.dumps(report, indent=2) if report is not None else "No report found")

{
  "match_id": 9079737909,
  "table_rows": {
    "matches": 1,
    "players": 10,
    "player_stats": 15080,
    "positions": 14570,
    "damage_events": 8783,
    "purchases": 200,
    "deaths": 37,
    "death_assists": 28,
    "final_inventory": 75
  },
  "findings": {
    "invalid_game_clock": {
      "rows": 0,
      "blocking": true
    },
    "excluded_resource_rows": {
      "rows": 0,
      "blocking": false
    },
    "duplicate_player_snapshots": {
      "rows": 0,
      "blocking": true
    },
    "unmapped_resource_rows": {
      "rows": 0,
      "blocking": true
    },
    "duplicate_team_snapshots": {
      "rows": 0,
      "blocking": true
    },
    "missing_team_snapshots": {
      "rows": 0,
      "blocking": true
    },
    "unknown_replication_marker": {
      "rows": 0,
      "blocking": true
    },
    "invalid_hero_player_ids": {
      "rows": 0,
      "blocking": true
    },
    "unmapped_hero_teams": {
      "rows": 0,
      "blocking": true
    },
    "damage